In [2]:
import pandas as pd

df = pd.read_csv('data/train.csv')
print(df.head())

   track_id timestamp_start_radar_utc timestamp_end_radar_utc  \
0   2537685       2023-09-05 08:59:18     2023-09-05 09:00:52   
1   2538937       2023-09-05 09:00:31     2023-09-05 09:02:25   
2   2539560       2023-09-05 09:01:13     2023-09-05 09:03:36   
3   2541207       2023-09-05 09:06:17     2023-09-05 09:06:49   
4   2541221       2023-09-05 09:06:20     2023-09-05 09:06:51   

                                          trajectory  \
0  01020000E0E61000005200000026EE676787321B403B57...   
1  01020000E0E6100000670000000364E067F9351B4030B8...   
2  01020000E0E61000007D00000040B7E1C160401B4045EC...   
3  01020000E0E61000001D000000E051BEFB3E371B40A070...   
4  01020000E0E61000001E000000F49EF3E880371B4050DF...   

                                     trajectory_time radar_bird_size  \
0  [0.0, 1.01982, 2.03911, 3.05854, 4.07798, 5.09...      Small bird   
1  [0.0, 1.01519, 2.02996, 4.06014, 5.07671, 7.10...      Small bird   
2  [0.0, 2.02202, 3.03262, 4.04231, 5.05144, 6.06...    

In [3]:
print(df.describe())

           track_id     airspeed        min_z        max_z  observation_id  \
count  2.601000e+03  2601.000000  2601.000000  2601.000000     2601.000000   
mean   1.650886e+08    15.347333    24.508836    67.977071     3212.531334   
std    2.388479e+08     3.516768    46.978121    55.832393     1571.854270   
min    2.537685e+06     5.280130    -9.841000     3.009000      267.000000   
25%    2.285202e+07    12.880600    -0.159000    31.186000     1805.000000   
50%    3.107417e+07    14.946000     5.386000    54.280000     3363.000000   
75%    5.268187e+08    17.436100    32.856000    85.897000     4891.000000   
max    5.823223e+08    35.371200   395.269000   437.180000     5580.000000   

       primary_observation_id  n_birds_observed  
count             2601.000000       2601.000000  
mean              3211.450211         11.269512  
std               1571.545717         34.244955  
min                267.000000          1.000000  
25%               1805.000000          1.000000

In [4]:
counts = df['bird_group'].value_counts()
print(counts)
print('')
percentages = df['bird_group'].value_counts(normalize=True) * 100
print(percentages)

bird_group
Gulls            1503
Songbirds         483
Pigeons           122
Waders            120
Birds of Prey     108
Clutter            84
Geese              83
Ducks              58
Cormorants         40
Name: count, dtype: int64

bird_group
Gulls            57.785467
Songbirds        18.569781
Pigeons           4.690504
Waders            4.613610
Birds of Prey     4.152249
Clutter           3.229527
Geese             3.191080
Ducks             2.229912
Cormorants        1.537870
Name: proportion, dtype: float64


In [5]:
# Convert timestamps
df["ts"] = pd.to_datetime(df["timestamp_start_radar_utc"])

# Extract hour and day of year
df["hour"] = df["ts"].dt.hour
df["day_of_year"] = df["ts"].dt.dayofyear

# Convert day_of_year -> seasons
def get_season(day):
    if 80 <= day < 172:
        return "spring"
    elif 172 <= day < 264:
        return "summer"
    elif 264 <= day < 355:
        return "autumn"
    else:
        return "winter"

df["season"] = df["day_of_year"].apply(get_season)

# -------------------------------
# Bird groups per hour
# -------------------------------

hour_table = pd.crosstab(df["hour"], df["bird_group"])

print("Bird groups per hour")
display(hour_table)

# -------------------------------
# Bird groups per season
# -------------------------------

season_table = pd.crosstab(df["season"], df["bird_group"])

print("\nBird groups per season")
display(season_table)

# -------------------------------
# Bird groups per season + hour
# -------------------------------

season_hour_table = pd.crosstab(
    [df["season"], df["hour"]],
    df["bird_group"]
)

print("\nBird groups per season and hour")
display(season_hour_table)

Bird groups per hour


bird_group,Birds of Prey,Clutter,Cormorants,Ducks,Geese,Gulls,Pigeons,Songbirds,Waders
hour,,,,,,,,,
6,0,0,0,0,0,1,0,4,0
7,7,11,4,7,18,75,2,43,6
8,20,10,8,9,11,103,2,31,11
9,24,21,7,12,16,223,6,83,22
10,22,33,4,22,17,384,4,129,38
11,19,9,4,5,4,239,5,36,19
12,10,0,8,1,3,149,1,55,8
13,1,0,0,0,2,76,6,41,5
14,3,0,1,1,10,137,75,30,6



Bird groups per season


bird_group,Birds of Prey,Clutter,Cormorants,Ducks,Geese,Gulls,Pigeons,Songbirds,Waders
season,,,,,,,,,
autumn,27,10,24,39,63,799,109,337,32
spring,49,70,1,18,10,215,5,55,50
summer,30,3,9,0,0,321,5,72,27
winter,2,1,6,1,10,168,3,19,11



Bird groups per season and hour


bird_group   Birds of Prey  Clutter  Cormorants  Ducks  Geese  Gulls  Pigeons  \
season hour                                                                     
autumn 6                 0        0           0      0      0      1        0   
       7                 0        0           4      3     13     52        1   
       8                 3        0           8      2     10     79        0   
       9                 7        0           4      8     15    148        4   
       10                9       10           2     20     14    226        1   
       11                5        0           3      4      0    130        3   
       12                0        0           1      1      0      1        1   
       13                0        0           0      0      1      9        5   
       14                3        0           1      0      8     76       74   
       15                0        0           1      1      2     77       20   
spring 7                 7       11           0      4      5     23        1   
       8                17       10           0      7      1     24        2   
       9                14       21           1      4      1     51        0   
       10                7       22           0      2      3    113        2   
       11                4        6           0      1      0      4        0   
summer 8                 0        0           0      0      0      0        0   
       9                 3        0           2      0      0     24        2   
       10                6        1           2      0      0     45        1   
       11               10        2           1      0      0     98        2   
       12               10        0           4      0      0    122        0   
       13                1        0           0      0      0     32        0   
winter 11                0        1           0      0      4      7        0   
       12                0        0           3      0      3     26        0   
       13                0        0           0      0      1     35        1   
       14                0        0           0      1      2     61        1   
       15                2        0           3      0      0     39        1   

bird_group   Songbirds  Waders  
season hour                     
autumn 6             4       0  
       7            40       4  
       8            14       2  
       9            62       4  
       10           90      15  
       11           24       7  
       12           19       0  
       13           41       0  
       14           25       0  
       15           18       0  
spring 7             3       2  
       8            17       8  
       9            11      17  
       10           23      23  
       11            1       0  
summer 8             0       1  
       9            10       1  
       10           16       0  
       11           11      12  
       12           35       8  
       13            0       5  
winter 11            0       0  
       12            1       0  
       13            0       0  
       14            5       6  
       15           13       5

In [6]:
print(df['radar_bird_size'].value_counts())

radar_bird_size
Small bird     1657
Large bird      449
Flock           345
Medium bird     150
Name: count, dtype: int64
